In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from model_utils import *

In [4]:
def generate_count_batch(batch_size, seq_len, vocab_size=5):
    X = torch.randint(0, vocab_size, (batch_size, seq_len))
    targets = torch.randint(0, vocab_size, (batch_size,))
    counts = (X == targets.unsqueeze(1)).sum(dim=1)
    return X, targets, counts

In [5]:
X, targets, counts = generate_count_batch(batch_size=1000, seq_len=6, vocab_size=5)
print(counts.float().mean().item())
print(torch.bincount(counts))

1.2009999752044678
tensor([247, 415, 243,  81,  13,   1])


In [7]:
def train_count_model(model, epochs, batch_size, seq_len, vocab_size=10, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    max_count = seq_len

    for epoch in range(epochs):
        X, targets, counts = generate_count_batch(batch_size, seq_len, vocab_size)
        model_input = torch.cat([targets.unsqueeze(1), X], dim=1)

        logits = model(model_input)
        first_pos_logits = logits[:, 0, :max_count+1]
        loss = loss_fn(first_pos_logits, counts)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 100 == 0:
            predictions = first_pos_logits.argmax(dim=-1)
            accuracy = (predictions == counts).float().mean()
            print(f"Epoch {epoch}: loss = {loss.item():.4f}, accuracy = {accuracy.item():.4f}")

    return model

In [9]:
model_count_sin = TinyTransformer(vocab_size=10, max_seq_len=30, d_model=32, num_heads=4, d_ff=128, num_layers=2)
model_count_sin = train_count_model(model_count_sin, epochs=1000, batch_size=64, seq_len=6, vocab_size=5)

Epoch 0: loss = 2.0890, accuracy = 0.1562
Epoch 100: loss = 1.2499, accuracy = 0.5312
Epoch 200: loss = 0.5942, accuracy = 0.7656
Epoch 300: loss = 0.2182, accuracy = 0.9375
Epoch 400: loss = 0.1227, accuracy = 0.9688
Epoch 500: loss = 0.0236, accuracy = 1.0000
Epoch 600: loss = 0.0258, accuracy = 1.0000
Epoch 700: loss = 0.0136, accuracy = 1.0000
Epoch 800: loss = 0.0058, accuracy = 1.0000
Epoch 900: loss = 0.0066, accuracy = 1.0000


In [10]:
def evaluate_count_at_lengths(model, lengths, vocab_size=5, batch_size=200):
    model.eval()
    results = {}
    with torch.no_grad():
        for length in lengths:
            X_test, targets_test, counts_test = generate_count_batch(batch_size, length, vocab_size)
            model_input = torch.cat([targets_test.unsqueeze(1), X_test], dim=1)
            logits = model(model_input)
            max_count = length
            first_pos_logits = logits[:, 0, :max_count+1]
            predictions = first_pos_logits.argmax(dim=-1)
            accuracy = (predictions == counts_test).float().mean().item()
            results[length] = accuracy
    model.train()
    return results

In [11]:
model_count_sin = TinyTransformer(vocab_size=30, max_seq_len=30, d_model=32, num_heads=4, d_ff=128, num_layers=2)
model_count_sin = train_count_model(model_count_sin, epochs=1000, batch_size=64, seq_len=6, vocab_size=5)

Epoch 0: loss = 2.0970, accuracy = 0.1094
Epoch 100: loss = 1.2871, accuracy = 0.3594
Epoch 200: loss = 0.4369, accuracy = 0.8750
Epoch 300: loss = 0.2060, accuracy = 0.9062
Epoch 400: loss = 0.3245, accuracy = 0.8906
Epoch 500: loss = 0.0497, accuracy = 0.9844
Epoch 600: loss = 0.0556, accuracy = 0.9844
Epoch 700: loss = 0.0542, accuracy = 0.9844
Epoch 800: loss = 0.0179, accuracy = 1.0000
Epoch 900: loss = 0.0089, accuracy = 1.0000


In [12]:
lengths_to_test = [6, 8, 10, 12, 15, 20, 25]
count_sin_results = evaluate_count_at_lengths(model_count_sin, lengths_to_test)
print(count_sin_results)

{6: 0.9950000047683716, 8: 0.27000001072883606, 10: 0.125, 12: 0.07000000029802322, 15: 0.029999999329447746, 20: 0.014999999664723873, 25: 0.004999999888241291}


In [14]:
model_count_sin.eval()
X_test, targets_test, counts_test = generate_count_batch(20, 25, vocab_size=5)
model_input = torch.cat([targets_test.unsqueeze(1), X_test], dim=1)
with torch.no_grad():
    logits = model_count_sin(model_input)
    predictions = logits[:, 0, :26].argmax(dim=-1)
print("True counts:   ", counts_test.tolist())
print("Predicted:     ", predictions.tolist())
model_count_sin.train()

True counts:    [2, 4, 6, 4, 4, 8, 7, 7, 7, 4, 3, 3, 4, 5, 8, 8, 6, 5, 4, 7]
Predicted:      [0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1]


TinyTransformer(
  (embedding): TokenAndPositionEmbedding(
    (token_embed): Embedding(30, 32)
  )
  (blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (attention): MultiHeadAttention(
        (W_q): Linear(in_features=32, out_features=32, bias=True)
        (W_k): Linear(in_features=32, out_features=32, bias=True)
        (W_v): Linear(in_features=32, out_features=32, bias=True)
        (W_o): Linear(in_features=32, out_features=32, bias=True)
      )
      (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
      (feed_forward): FeedForward(
        (linear1): Linear(in_features=32, out_features=128, bias=True)
        (relu): ReLU()
        (linear2): Linear(in_features=128, out_features=32, bias=True)
      )
      (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
    )
  )
  (output_layer): Linear(in_features=32, out_features=30, bias=True)
)

In [15]:
lengths_check = [3, 4, 5, 6]
short_check = evaluate_count_at_lengths(model_count_sin, lengths_check)
print(short_check)

{3: 0.23000000417232513, 4: 0.41499999165534973, 5: 0.9399999976158142, 6: 1.0}


In [16]:
def generate_count_batch_variable_length(batch_size, min_len, max_len, vocab_size=5):
    seq_len = torch.randint(min_len, max_len + 1, (1,)).item()
    return generate_count_batch(batch_size, seq_len, vocab_size)

In [17]:
def train_count_model_variable(model, epochs, batch_size, min_len, max_len, vocab_size=5, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    max_count = max_len

    for epoch in range(epochs):
        X, targets, counts = generate_count_batch_variable_length(batch_size, min_len, max_len, vocab_size)
        model_input = torch.cat([targets.unsqueeze(1), X], dim=1)

        logits = model(model_input)
        first_pos_logits = logits[:, 0, :max_count+1]
        loss = loss_fn(first_pos_logits, counts)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 100 == 0:
            predictions = first_pos_logits.argmax(dim=-1)
            accuracy = (predictions == counts).float().mean()
            print(f"Epoch {epoch}: loss = {loss.item():.4f}, accuracy = {accuracy.item():.4f}")

    return model

In [18]:
model_count_sin = TinyTransformer(vocab_size=30, max_seq_len=30, d_model=32, num_heads=4, d_ff=128, num_layers=2)
model_count_sin = train_count_model_variable(model_count_sin, epochs=1000, batch_size=64, min_len=3, max_len=6, vocab_size=5)

Epoch 0: loss = 2.3940, accuracy = 0.0156
Epoch 100: loss = 1.4174, accuracy = 0.3125
Epoch 200: loss = 1.0375, accuracy = 0.4531
Epoch 300: loss = 0.1778, accuracy = 0.9688
Epoch 400: loss = 0.1177, accuracy = 0.9688
Epoch 500: loss = 0.0372, accuracy = 1.0000
Epoch 600: loss = 0.0954, accuracy = 0.9688
Epoch 700: loss = 0.0132, accuracy = 1.0000
Epoch 800: loss = 0.0159, accuracy = 1.0000
Epoch 900: loss = 0.0199, accuracy = 1.0000


In [19]:
lengths_to_test = [3, 4, 5, 6, 8, 10, 12, 15, 20, 25]
count_sin_results = evaluate_count_at_lengths(model_count_sin, lengths_to_test, vocab_size=5)
print(count_sin_results)

{3: 1.0, 4: 1.0, 5: 0.9950000047683716, 6: 0.9700000286102295, 8: 0.6899999976158142, 10: 0.17000000178813934, 12: 0.07000000029802322, 15: 0.029999999329447746, 20: 0.02500000037252903, 25: 0.004999999888241291}


In [20]:
model_count_nope = TinyTransformerNoPE(vocab_size=30, d_model=32, num_heads=4, d_ff=128, num_layers=2)
model_count_nope = train_count_model_variable(model_count_nope, epochs=1000, batch_size=64, min_len=3, max_len=6, vocab_size=5)

model_count_learned = TinyTransformerLearnedPE(vocab_size=30, max_seq_len=30, d_model=32, num_heads=4, d_ff=128, num_layers=2)
model_count_learned = train_count_model_variable(model_count_learned, epochs=1000, batch_size=64, min_len=3, max_len=6, vocab_size=5)

Epoch 0: loss = 2.1274, accuracy = 0.1406
Epoch 100: loss = 1.2012, accuracy = 0.5625
Epoch 200: loss = 0.7854, accuracy = 0.6875
Epoch 300: loss = 0.6405, accuracy = 0.6250
Epoch 400: loss = 0.3028, accuracy = 0.9531
Epoch 500: loss = 0.5278, accuracy = 0.7344
Epoch 600: loss = 0.4335, accuracy = 0.7656
Epoch 700: loss = 0.3515, accuracy = 0.8125
Epoch 800: loss = 0.2112, accuracy = 0.9062
Epoch 900: loss = 0.2625, accuracy = 0.9219
Epoch 0: loss = 2.0182, accuracy = 0.2500
Epoch 100: loss = 1.2619, accuracy = 0.4688
Epoch 200: loss = 0.3488, accuracy = 0.8281
Epoch 300: loss = 0.0795, accuracy = 0.9688
Epoch 400: loss = 0.1602, accuracy = 0.9219
Epoch 500: loss = 0.0462, accuracy = 0.9844
Epoch 600: loss = 0.0079, accuracy = 1.0000
Epoch 700: loss = 0.0028, accuracy = 1.0000
Epoch 800: loss = 0.0121, accuracy = 1.0000
Epoch 900: loss = 0.0018, accuracy = 1.0000


In [21]:
lengths_to_test = [3, 4, 5, 6, 8, 10, 12, 15, 20, 25]

count_nope_results = evaluate_count_at_lengths(model_count_nope, lengths_to_test, vocab_size=5)
print("NoPE:", count_nope_results)

count_learned_results = evaluate_count_at_lengths(model_count_learned, lengths_to_test, vocab_size=5)
print("Learned PE:", count_learned_results)

NoPE: {3: 0.9350000023841858, 4: 0.9549999833106995, 5: 0.8199999928474426, 6: 0.6000000238418579, 8: 0.2150000035762787, 10: 0.07999999821186066, 12: 0.0949999988079071, 15: 0.019999999552965164, 20: 0.03999999910593033, 25: 0.009999999776482582}
Learned PE: {3: 1.0, 4: 1.0, 5: 1.0, 6: 0.9950000047683716, 8: 0.7350000143051147, 10: 0.5099999904632568, 12: 0.29499998688697815, 15: 0.125, 20: 0.05999999865889549, 25: 0.02500000037252903}
